# GeoTerra · WebGIS do limão em São Paulo

Execute as células em ordem. O projeto gera um mapa Folium publicável no GitHub Pages. A cópia inicial usa dados oficiais de 2020–2024; os anos posteriores ficam desativados enquanto não publicados. Preços e comercialização ainda não foram integrados.

## 1. Preparar o projeto
Baixa os arquivos públicos para uma pasta temporária nova, sem sobrescrever seu trabalho.

In [ ]:
import tempfile, subprocess, os, sys
from pathlib import Path
pasta = Path(tempfile.mkdtemp(prefix='geoterra_limao_')) / 'projeto'
subprocess.run(['git', 'clone', 'https://github.com/geovanemariotto123-coder/-webgis-limao-sao-paulo.git', str(pasta)], check=True)
os.chdir(pasta)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## 2. Escolher a atualização
O padrão utiliza os dados preservados no projeto. Troque para `True` para consultar novamente a PAM. Erros de conexão interrompem a atualização, sem substituir dados ausentes por números inventados.

In [ ]:
ATUALIZAR_IBGE = False
if ATUALIZAR_IBGE:
    from coletar import coletar
    coletar()

## 3. Gerar e conferir
O gerador confere códigos municipais, duplicatas e totais estaduais de produção e área. Valores monetários podem diferir por arredondamento. Os registros da conferência ficam em `data/validacao.json`.

In [ ]:
from gerar import gerar
mapa = gerar()
import json
print(json.loads(Path('data/fontes.json').read_text())['anos'])

## 4. Visualizar
Selecione o ano e indicador, busque um município e clique no mapa para abrir sua série histórica.

In [ ]:
from IPython.display import HTML, display
import html
conteudo = Path('index.html').read_text(encoding='utf-8')
display(HTML('<iframe srcdoc="'+html.escape(conteudo, quote=True)+'" style="width:100%;height:850px;border:0"></iframe>'))

## 5. Baixar e publicar
Baixe o HTML e coloque-o na raiz do repositório com o nome `index.html`. No GitHub: Settings → Pages → Deploy from a branch → main → / (root) → Save. O CSV completo acompanha o ZIP, incluindo símbolos originais da fonte.

In [ ]:
import zipfile
with zipfile.ZipFile('WebGIS_Limao_SP.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for nome in ['index.html', 'data/limao_sp.csv', 'data/fontes.json', 'data/validacao.json']:
        z.write(nome)
try:
    from google.colab import files
    files.download('WebGIS_Limao_SP.zip')
except ImportError:
    print('Arquivo disponível em:', Path('WebGIS_Limao_SP.zip').resolve())

## Fontes e limites
[IBGE/PAM, tabela 1613](https://sidra.ibge.gov.br/tabela/1613). “Limão” não separa cultivares. Área destinada à colheita não equivale a novos plantios. Valor da produção é nominal, em mil R$, e não mede cotação nem comercialização. A malha simplificada atual serve à visualização municipal, não a delimitação de propriedades. Os dados originais e URLs estão em `data/`.